# Telemetry Parsing and Inspection — CDL-DET-V2

This notebook inspects the Mordor adversary emulation dataset (`empire_mimikatz_logonpasswords_2020-08-07103224.json`), verifies JSON integrity, handles path discovery across execution contexts, and summarizes security events by Event ID and Channel.

In [1]:
from pathlib import Path
import json
import pandas as pd

# Robust relative-path resolution: checks current directory and parent data/ directory
candidate_paths = [
    Path("empire_mimikatz_logonpasswords_2020-08-07103224.json"),
    Path("data") / "empire_mimikatz_logonpasswords_2020-08-07103224.json",
    Path("..") / "data" / "empire_mimikatz_logonpasswords_2020-08-07103224.json",
]

dataset_path = None
for p in candidate_paths:
    if p.is_file():
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError(
        f"❌ Dataset not found! Checked relative paths: {[str(p) for p in candidate_paths]}. "
        f"Current working directory: {Path.cwd()}"
    )

print(f"✅ Successfully located dataset at: {dataset_path.resolve()}")
print(f"   Current working directory: {Path.cwd()}")

✅ Successfully located dataset at: C:\Users\Hala\Downloads\Project Files\cdl-det-v2-main\data\empire_mimikatz_logonpasswords_2020-08-07103224.json
   Current working directory: C:\Users\Hala\Downloads\Project Files\cdl-det-v2-main\data


In [2]:
# Streaming parse of NDJSON lines with explicit malformed line tracking
events = []
malformed_lines = []
total_lines = 0

with open(dataset_path, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        total_lines += 1
        line_clean = line.strip()
        if not line_clean:
            continue
        try:
            record = json.loads(line_clean)
            events.append({
                "EventID": record.get("EventID"),
                "Channel": record.get("Channel"),
                "UtcTime": record.get("UtcTime") or record.get("@timestamp"),
                "Hostname": record.get("Hostname"),
                "Image": record.get("Image") or record.get("NewProcessName"),
                "TargetImage": record.get("TargetImage"),
                "GrantedAccess": record.get("GrantedAccess"),
                "SourceImage": record.get("SourceImage")
            })
        except json.JSONDecodeError as err:
            malformed_lines.append((line_num, str(err)))

df = pd.DataFrame(events)
print(f"✅ Total lines scanned: {total_lines}")
print(f"✅ Valid events loaded: {len(df)}")
if malformed_lines:
    print(f"⚠️ Malformed JSON lines encountered: {len(malformed_lines)}")
    for l_no, err_msg in malformed_lines[:5]:
        print(f"   Line {l_no}: {err_msg}")
else:
    print("✅ Zero malformed JSON lines detected.")

✅ Total lines scanned: 6026
✅ Valid events loaded: 6026
✅ Zero malformed JSON lines detected.


In [3]:
# Group and display event counts by Channel and Event ID
summary = df.groupby(["Channel", "EventID"]).size().reset_index(name="Count")
summary = summary.sort_values(by="Count", ascending=False).reset_index(drop=True)

print("📊 Telemetry Event Distribution Summary:")
print(summary.to_string(index=False))

📊 Telemetry Event Distribution Summary:
                                   Channel  EventID  Count
                        Windows PowerShell      800   2940
  Microsoft-Windows-PowerShell/Operational     4103   1695
      Microsoft-Windows-Sysmon/Operational       10    272
                                  Security     5156    211
                                  Security     5158    195
      Microsoft-Windows-Sysmon/Operational       12    154
      Microsoft-Windows-Sysmon/Operational        3     90
                                  Security     4658     86
      Microsoft-Windows-Sysmon/Operational        7     81
                                  Security     4656     49
      Microsoft-Windows-Sysmon/Operational       13     45
                                  Security     4663     43
                                  Security     4690     43
      Microsoft-Windows-Sysmon/Operational       11     14
                                  Security     4703     13
                

In [4]:
# Verify key detection indicators in the dataset
lsass_events = df[(df["EventID"] == 10) & (df["TargetImage"].str.contains("lsass", case=False, na=False))]
print(f"🔍 Sysmon Event ID 10 (LSASS Handle Access): {len(lsass_events)} record(s)")
for _, row in lsass_events.iterrows():
    print(f"   Source: {row['SourceImage']} | Access: {row['GrantedAccess']} | Target: {row['TargetImage']}")

whoami_events = df[(df["EventID"] == 1) & (df["Image"].str.contains("whoami", case=False, na=False))]
print(f"\n🔍 Sysmon Event ID 1 (Whoami Execution): {len(whoami_events)} record(s)")
for _, row in whoami_events.iterrows():
    print(f"   Image: {row['Image']} at {row['UtcTime']}")

ps_events = df[df["Channel"] == "Microsoft-Windows-PowerShell/Operational"]
print(f"\n🔍 PowerShell Operational Log Events: {len(ps_events)} record(s)")

🔍 Sysmon Event ID 10 (LSASS Handle Access): 4 record(s)
   Source: C:\windows\system32\svchost.exe | Access: 0x1000 | Target: C:\windows\system32\lsass.exe
   Source: C:\windows\system32\svchost.exe | Access: 0x1000 | Target: C:\windows\system32\lsass.exe
   Source: C:\windows\System32\WindowsPowerShell\v1.0\powershell.exe | Access: 0x1010 | Target: C:\windows\system32\lsass.exe
   Source: C:\windows\system32\svchost.exe | Access: 0x2000 | Target: C:\windows\system32\lsass.exe

🔍 Sysmon Event ID 1 (Whoami Execution): 1 record(s)
   Image: C:\Windows\System32\whoami.exe at 2020-08-07 14:32:45.881

🔍 PowerShell Operational Log Events: 1695 record(s)
